# Train Decision Segmenter (OPF) — v5 Label Space

Fine-tune the OpenAI Privacy Filter (1.5B params) as a 20-class judicial decision segmenter.

**Data paths (in priority order):**
1. **Reviewed splits (fastest):** Use `data/segmenter_splits/` from repo (subagent-reviewed person spans)
2. **IA download:** Download prepared splits from Internet Archive
3. **Full pipeline:** Download texts from IA, run OPF + heuristic labeling from scratch

**Requirements:** GPU runtime (T4 or better, 15GB+ VRAM)

Go to **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Switch to GPU runtime first!"

In [ ]:
!uv pip install -q "opf @ git+https://github.com/openai/privacy-filter.git" httpx structlog

In [ ]:
# Download OPF base checkpoint (~2.8GB)
from opf._common.checkpoint_download import ensure_default_checkpoint
ensure_default_checkpoint()

In [ ]:
# Clone repo
!git clone --depth 1 https://github.com/franklinbaldo/causaganha.git /content/causaganha

import os
os.chdir("/content/causaganha")
os.environ["PYTHONPATH"] = "/content/causaganha"

!uv pip install -q ibis-framework[duckdb] structlog pyarrow

## Option A: Use reviewed splits from repo (fastest)

The repo includes subagent-reviewed training splits in `data/segmenter_splits/`.
These have person spans classified by Claude subagents — highest quality data.

In [ ]:
import os, shutil

MODEL_DIR = "/content/models/decision_segmenter"
REPO_SPLITS = "/content/causaganha/data/segmenter_splits"
os.makedirs(MODEL_DIR, exist_ok=True)

if os.path.exists(f"{REPO_SPLITS}/train.jsonl"):
    for f in ["train.jsonl", "val.jsonl", "test.jsonl", "label_space.json"]:
        shutil.copy(f"{REPO_SPLITS}/{f}", f"{MODEL_DIR}/{f}")
    for split in ["train", "val", "test"]:
        count = sum(1 for _ in open(f"{MODEL_DIR}/{split}.jsonl"))
        print(f"  {split}: {count} examples")
    print("\nReviewed splits ready — skip to 'Train with OPF' below")
else:
    print("No reviewed splits in repo — try Option A2 or A3 below")

## Option A2: Download prepared splits from IA

Falls back to downloading from Internet Archive if repo splits aren't available.

In [ ]:
import os, subprocess, tarfile

MODEL_DIR = "/content/models/decision_segmenter"
os.makedirs(MODEL_DIR, exist_ok=True)

SPLITS_URL = "https://archive.org/download/causaganha-segmenter-data/segmenter_training_data.tar.gz"
SPLITS_TAR = f"{MODEL_DIR}/splits.tar.gz"

result = subprocess.run(
    ["curl", "-fL", SPLITS_URL, "-o", SPLITS_TAR],
    capture_output=True
)

if result.returncode == 0 and os.path.exists(SPLITS_TAR):
    with tarfile.open(SPLITS_TAR) as tf:
        tf.extractall(MODEL_DIR)
    os.remove(SPLITS_TAR)
    for split in ["train", "val", "test"]:
        path = f"{MODEL_DIR}/{split}.jsonl"
        if os.path.exists(path):
            count = sum(1 for _ in open(path))
            print(f"  {split}: {count} examples")
    print("\nSplits ready — skip to 'Train with OPF' below")
else:
    print("No prepared splits on IA — try Option B below")

## Option B: Full bootstrap pipeline

Downloads texts from IA, runs OPF base model for entity detection,
applies heuristic section labels, and prepares training splits.
Takes ~15-20 min on T4.

In [ ]:
MODEL_DIR = "/content/models/decision_segmenter"

# Step 1: Download texts from IA (stratified by tribunal)
!uv run python scripts/augment_segmenter_data.py \
    --target 2000 \
    --max-per-tribunal 150 \
    --output-dir {MODEL_DIR} \
    --n-items 10

In [ ]:
# Step 2: OPF entity detection + heuristic sections + auto-map
# Uses GPU for OPF inference (much faster than CPU)
MODEL_DIR = "/content/models/decision_segmenter"

!uv run python scripts/bootstrap_training_corpus.py \
    --input {MODEL_DIR}/train.jsonl \
    --output-dir {MODEL_DIR} \
    --target 2000 \
    --device 0 \
    --skip-haiku

## Train with OPF

Uses `opf train` (native CLI) to fine-tune the 1.5B model.

**Memory tips:**
- T4 (15GB): `--batch-size 1 --n-ctx 512`
- A100 (40GB): `--batch-size 4 --n-ctx 1024`

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_DIR = "/content/models/decision_segmenter"

!uv run python -m opf train {MODEL_DIR}/train.jsonl \
    --validation-dataset {MODEL_DIR}/val.jsonl \
    --label-space-json {MODEL_DIR}/label_space.json \
    --output-dir {MODEL_DIR}/best \
    --device cuda \
    --epochs 3 \
    --batch-size 1 \
    --n-ctx 512

## Evaluate

In [ ]:
MODEL_DIR = "/content/models/decision_segmenter"

!uv run python -m opf eval {MODEL_DIR}/test.jsonl \
    --checkpoint {MODEL_DIR}/best \
    --device cuda \
    --per-class \
    --metrics-out {MODEL_DIR}/test_metrics.json

In [ ]:
import json

MODEL_DIR = "/content/models/decision_segmenter"
metrics = json.load(open(f"{MODEL_DIR}/test_metrics.json"))

print("=" * 60)
print("TEST METRICS (v5 label space — 20 classes)")
print("=" * 60)
macro = metrics.get("macro avg", {})
print(f"Macro F1: {macro.get('f1-score', 0):.3f}")
disp = metrics.get("sec_dispositivo", {})
print(f"sec_dispositivo F1: {disp.get('f1-score', 0):.3f}")
print()
for k, v in metrics.items():
    if isinstance(v, dict) and "f1-score" in v and k not in ("macro avg", "weighted avg", "micro avg"):
        print(f"  {k:<22} P={v.get('precision',0):.2f}  R={v.get('recall',0):.2f}  F1={v.get('f1-score',0):.2f}  n={v.get('support',0)}")

## Download trained model

In [ ]:
MODEL_DIR = "/content/models/decision_segmenter"

!tar -czf /content/decision_segmenter_v5.tar.gz -C {MODEL_DIR}/best .
print(f"Model saved: /content/decision_segmenter_v5.tar.gz")

# Optional: mount Drive and copy
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/decision_segmenter_v5.tar.gz /content/drive/MyDrive/
# !cp {MODEL_DIR}/test_metrics.json /content/drive/MyDrive/